In [1]:
#perform 70-30 test-validation within each geospatial clusters 
#for each cluster we select random 70% of stations as training and 30% as validation
#final dataframes: train_stn: stn_id, cluster_id
#                  val_stn: stn_id, cluster_id

In [2]:
import pandas as pd
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import rioxarray as rxr
import rasterio as rio
import os
from sklearn.cluster import DBSCAN

In [3]:
#load stations and cluster datasets
save_dir = r"../Data/Processed/Split_stations/"
prec_cluster_path = r"..\Data\Processed\Spatial_clustering\precipitation_clusters.tif"
insitu_prec_df = pd.read_csv(r"..\Data\Processed\Insitu\insitu_rainfall_1991_2020.csv")

stn_df = insitu_prec_df.drop_duplicates(subset=['index_no'], keep='first').reset_index(drop=True)
stn_df = stn_df[['index_no','lat', 'long']]

In [4]:
#sample station points to extract cluster value
stn_points = list(zip(stn_df['long'], stn_df['lat']))

#extract cluster value
cluster_values = []
with rio.open(prec_cluster_path) as src:
    sampled_values = list(src.sample(stn_points))
    for i, value in enumerate(sampled_values):
        cluster_values.append(int(value[0]))
         
#add cluster values to station df
stn_df['cluster_id'] = cluster_values

stn_df.to_csv(save_dir + r"\clustered_stations.csv", index=False)

In [5]:
coords = np.radians(stn_df[['lat', 'long']].to_numpy())

kms_per_radian = 6371.0088
search_distance_km = 12  
epsilon = search_distance_km / kms_per_radian
min_samples_per_cluster = 2

db = DBSCAN(eps=epsilon, min_samples=min_samples_per_cluster, algorithm='ball_tree', metric='haversine').fit(coords)
stn_df['db_cluster'] = db.labels_

# Count number of stations with db_cluster = -1 (noise)
noise_count = stn_df[stn_df['db_cluster'] == -1].shape[0]
print(f"Number of noise stations: {noise_count}")

# Determine the start of the new labels to avoid collision with existing clusters
current_max_label = stn_df['db_cluster'].max()
start_label = current_max_label + 1
end_label = start_label + noise_count 

available_labels = np.arange(start_label, end_label)

# Assign random unique db_cluster to db_cluster = -1 (noise)
unique_noise_labels = np.random.choice(available_labels, size=noise_count, replace=False)
stn_df.loc[stn_df['db_cluster'] == -1, 'db_cluster'] = unique_noise_labels

stn_df

Number of noise stations: 53


,index_no,lat,long,cluster_id,db_cluster
0,102,29.55744,80.41317,0,28
1,103,29.46714,80.54575,0,9
2,104,29.30135,80.58775,0,36
3,105,28.95477,80.23049,0,5
4,107,29.84496,80.54539,0,57
...,...,...,...,...,...
63,1407,26.90964,87.92308,1,26
64,1408,26.67067,87.70318,0,25
65,1412,26.56389,88.06150,1,33
66,1416,26.86806,88.07833,1,31


In [6]:
#Split stations within each cluster into train and test
cluster_ids = np.unique(cluster_values)
train_df = pd.DataFrame()
test_df = pd.DataFrame()
split_fraction = 0.7

for cluster_id in cluster_ids:
    cluster_stations = stn_df[stn_df['cluster_id'] == cluster_id].set_index('db_cluster')
    unique_db_clusters = cluster_stations.index.unique()
    train_clusters = pd.Series(unique_db_clusters).sample(frac=split_fraction, random_state = 41)
    
    train_df_lv = cluster_stations.loc[train_clusters].reset_index()
    test_df_lv = cluster_stations.drop(train_clusters).reset_index()
    
    train_df = pd.concat([train_df, train_df_lv], ignore_index=True)
    test_df = pd.concat([test_df, test_df_lv], ignore_index=True)


#save both splits
save_dir = r"../Data/Processed/Split_stations/"
os.makedirs(save_dir, exist_ok=True)
train_df.to_csv(save_dir + 'train_stations.csv', index=False)
test_df.to_csv(save_dir + 'test_stations.csv', index=False)

In [8]:
# %%
# Calculate inter-station distance matrix and find exclusive nearest-neighbor pairs
from sklearn.metrics.pairwise import haversine_distances

# 1. Calculate the distance matrix
train_coords_rad = np.radians(train_df[['lat', 'long']].to_numpy())
dist_matrix_rad = haversine_distances(train_coords_rad, train_coords_rad)

earth_radius_km = 6371.0088
dist_matrix_km = dist_matrix_rad * earth_radius_km

# 2. Prepare the matrix for greedy pairing
# We work with a copy and set the diagonal to infinity so a station doesn't pair with itself
dist_matrix_masked = dist_matrix_km.copy()
np.fill_diagonal(dist_matrix_masked, np.inf)

paired_distances = []
paired_stations = []

n_stations = dist_matrix_masked.shape[0]
n_pairs = n_stations // 2  # This will leave 1 station out if the total count is odd

# 3. Iteratively find the closest available pair
for _ in range(n_pairs):
    # Find the row and column index of the current minimum distance
    min_index = np.unravel_index(np.argmin(dist_matrix_masked), dist_matrix_masked.shape)
    i, j = min_index
    
    # Record the distance and the station IDs
    min_dist = dist_matrix_masked[i, j]
    paired_distances.append(min_dist)
    
    station_i = train_df.iloc[i]['index_no']
    station_j = train_df.iloc[j]['index_no']
    paired_stations.append((station_i, station_j))
    
    # "Remove" these two stations from future pairings by setting their rows and cols to infinity
    dist_matrix_masked[i, :] = np.inf
    dist_matrix_masked[:, i] = np.inf
    dist_matrix_masked[j, :] = np.inf
    dist_matrix_masked[:, j] = np.inf

# 4. Calculate and display the results
average_neighbor_distance = np.mean(paired_distances)

print(f"Total pairs formed: {len(paired_distances)}")
print(f"Average exclusive nearest-neighbor distance: {average_neighbor_distance:.2f} km")


Total pairs formed: 25
Average exclusive nearest-neighbor distance: 54.66 km
